# ฝึกโมเดลตรวจจับไฟและควัน — SeniorINC

Notebook เดียวแทนไฟล์ Python ทั้ง 5 ไฟล์ ไม่ต้องเรียก `.py` เพิ่ม
สำหรับ Windows 11 + NVIDIA RTX 4050 Laptop GPU

**วิธีใช้:** วาง Notebook และ ZIP ข้อมูลในโฟลเดอร์เดียวกัน เลือก kernel `Python (SeniorINC)`
แล้วรันทีละเซลล์จากบนลงล่างด้วย **Shift + Enter** ค่าที่ต้องแก้อยู่ในเซลล์ตั้งค่า
แต่ละขั้นจะบอกว่าต้องทำอะไรและจะได้ผลอะไร

**สถานะข้อมูลของคุณ:** มี 988 ภาพ แบ่ง Train 692 / Validation 197 / Test 99
มีกรอบไฟ/ควัน 2,302 รายการ แต่ `segmentation` ว่างทั้งหมด
จึงฝึกตรวจจับกรอบได้ทันที ส่วนการฝึก mask ต้องเพิ่ม polygon ที่ตรวจแก้แล้วก่อน
Notebook จะไม่สร้าง mask สี่เหลี่ยมจากกรอบแทนขอบวัตถุจริง

แนวทางใน [SeniorINC](https://github.com/Peanut2547/SeniorINC) ใช้ YOLO11-seg เพื่อให้ได้กรอบและ mask
ในโมเดลเดียว และนำ mask ไปวิเคราะห์พื้นที่/การเปลี่ยนแปลงตามเวลาและ depth ต่อ
โค้ดนี้เตรียมการ Train บนโน้ตบุ๊ก ยังไม่ใช่การ export/compile สำหรับ Hailo

Notebook นี้ยังไม่ได้รัน Train บน Windows หรือ GPU ของคุณ

## 0. เตรียม Miniconda และเปิด Jupyter — ทำครั้งแรกใน Miniconda Prompt

หากยังไม่ได้สร้าง environment ให้รันคำสั่งนี้ใน **Miniconda Prompt / Anaconda Prompt**
คำสั่งในกรอบนี้ไม่ใช่เซลล์ Python

```bat
conda create -n seniorinc_train python=3.11 -y
conda activate seniorinc_train
python -m pip install --upgrade pip
python -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
python -m pip install "ultralytics>=8.3,<9" jupyterlab ipykernel matplotlib
python -m ipykernel install --user --name seniorinc_train --display-name "Python (SeniorINC)"
cd /d D:\SeniorINC_Training
jupyter lab
```

เปลี่ยน `D:\SeniorINC_Training` ให้ตรงกับโฟลเดอร์ที่วางไฟล์จริง
หากมี environment `seniorinc_train` จากชุดเดิมแล้ว ให้ข้ามการ create และการติดตั้ง PyTorch ซ้ำ
เพียง activate แล้วติดตั้ง JupyterLab, ipykernel, matplotlib และลงทะเบียน kernel

ครั้งถัดไปใช้ `conda activate seniorinc_train` → เข้าโฟลเดอร์ → `jupyter lab`
เลือก kernel **Python (SeniorINC)** ที่มุมขวาบนของ Notebook
หาก CUDA ติดตั้งไม่ผ่าน ให้ตรวจ `nvidia-smi` และคำสั่งสำหรับ Windows/Pip/CUDA ที่
[PyTorch](https://pytorch.org/get-started/locally/) ก่อน

## 1. ตรวจว่า Notebook ใช้ Python และ GPU ถูกตัว

ต้องเห็น `CUDA available: True` และชื่อ RTX 4050 หากเป็น False เซลล์จะหยุดก่อนเริ่ม Train
ไม่ต้องติดตั้ง ROS2, DepthAI หรือ Hailo ใน environment สำหรับ Train นี้

In [1]:
import sys
from pathlib import Path
import torch
from ultralytics import YOLO

print('Python executable:', sys.executable)
print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('ยังใช้ CUDA ไม่ได้: ตรวจ kernel, NVIDIA driver และ CUDA-enabled PyTorch ก่อน')
gpu = torch.cuda.get_device_properties(0)
print('GPU:', gpu.name)
print(f'VRAM: {gpu.total_memory / 1024**3:.2f} GiB')
print('GPU calculation:', torch.ones(16, device='cuda').sum().item())


WARNING Ultralytics settings updated to the latest schema. Existing values were preserved where possible. 
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\phayu\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Python executable: c:\Users\phayu\miniconda3\envs\seniorinc\python.exe
PyTorch: 2.7.1+cu118
CUDA build: 11.8
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.00 GiB
GPU calculation: 16.0


## 2. ตั้งค่า — แก้หลัก ๆ ที่เซลล์นี้

เลือก `TASK` ตาม label ที่มี:

| TASK | ผลลัพธ์ | ใช้ ZIP ปัจจุบันได้หรือไม่ |
|---|---|---|
| `"detect"` | กรอบ + ชื่อ fire/smoke + confidence | ได้ |
| `"segment"` | กรอบ + ชื่อ fire/smoke + confidence + mask | ต้องเพิ่ม polygon ก่อน |

ค่าเริ่มต้นเป็น `detect` เพื่อทดลองกับข้อมูลที่คุณส่งมาได้ทันที
เมื่อต้องการฝึกตามเป้าหมายเต็ม ให้เปลี่ยนเป็น `segment` และเปลี่ยน ZIP เป็นชุดที่มี mask
ไม่จำเป็นต้องฝึก detect ให้เสร็จก่อนฝึก segment

เริ่ม `EPOCHS=1` เพื่อตรวจว่ารันได้ แล้วเปลี่ยนเป็น 100 และรันขั้น Train ใหม่เพื่อฝึกจริง
ใช้ batch 4 ก่อน ถ้าขึ้น CUDA out of memory ให้ลดเป็น 2 หรือ 1
RAM ในภาพเดิมใช้ไป 94% ควรปิดโปรแกรมหนักก่อนเริ่ม

In [ ]:
# โฟลเดอร์เดียวกับที่เปิด Notebook; เปลี่ยนเป็น Path(r"D:\SeniorINC_Training") ได้
WORK_DIR = Path.cwd().resolve()

TASK = "detect"  # เปลี่ยนเป็น "segment" เมื่อมี polygon masks ที่ตรวจแก้แล้ว
ZIP_PATH = WORK_DIR / "Fire and Smoke.coco.zip"
DATA_DIR = WORK_DIR / f"dataset_{TASK}_notebook"

EPOCHS = 1       # ทดสอบ 1 epoch ก่อน; ฝึกจริงเปลี่ยนเป็น 100
BATCH = 4
IMAGE_SIZE = 640
DEVICE = 0
RUN_NAME = f"fire_smoke_{TASK}"

assert TASK in ("detect", "segment")
assert ZIP_PATH.is_file(), f'ไม่พบ ZIP: {ZIP_PATH} — ตรวจชื่อไฟล์และ WORK_DIR'
print('งาน:', TASK)
print('ZIP:', ZIP_PATH)
print('ข้อมูลที่แปลงแล้ว:', DATA_DIR)
print('Epochs:', EPOCHS, '| batch:', BATCH, '| image size:', IMAGE_SIZE)


## 3. ตรวจ annotation ใน ZIP ก่อนแปลง

ดูจำนวนภาพ/กรอบของแต่ละชุดและจำนวน annotation ที่มี segmentation
หากเป็นศูนย์ทั้งหมด ให้เลือก detect หรือเพิ่ม mask ก่อน

In [ ]:
import json
import zipfile
from IPython.display import Markdown, display

summary = ['| ชุด | ภาพ | กรอบ fire | กรอบ smoke | มี segmentation |',
           '|---|---:|---:|---:|---:|']
with zipfile.ZipFile(ZIP_PATH) as archive:
    for member in sorted(archive.namelist()):
        if member.endswith('_annotations.coco.json'):
            data = json.loads(archive.read(member))
            names = {c['id']: c['name'] for c in data['categories']}
            anns = data['annotations']
            fire = sum(names[a['category_id']] == 'fire' for a in anns)
            smoke = sum(names[a['category_id']] == 'smoke' for a in anns)
            masks = sum(bool(a.get('segmentation')) for a in anns)
            split = member.rsplit('/', 1)[0]
            summary.append(f'| {split} | {len(data["images"])} | {fire} | {smoke} | {masks} |')
display(Markdown('\n'.join(summary)))


## 4. โหลดฟังก์ชันแปลงข้อมูล — รันอย่างเดียว ไม่ต้องแก้

เซลล์ยาวนี้รวมตัวแปลงจากไฟล์เดิมไว้ใน Notebook จึงไม่ต้องมีไฟล์ `.py` ข้างนอก
Jupyter อาจซ่อนโค้ดไว้ให้กดขยายได้

ตัวแปลงกำหนด `fire=0`, `smoke=1` และตัดภาพ Train ที่เหมือน Validation/Test แบบ byte-for-byte ออก
ชุดเดิมจะเหลือ Train 691 / Validation 197 / Test 99
ยังต้องตรวจภาพคล้ายหรือเฟรมจากเหตุการณ์เดียวกันเพิ่มเติม ไม่ใช่ตรวจได้จาก hash ทั้งหมด

In [ ]:
"""Convert this project's COCO ZIP to YOLO labels without inventing masks."""
import argparse
from collections import Counter, defaultdict
import hashlib
import json
import math
from pathlib import Path, PurePosixPath
import zipfile


def prepare(source, output, task):
    output = Path(output).resolve()
    if output.exists():
        raise ValueError('Output already exists. Choose a NEW --output directory.')
    rows, report = [], {}
    with zipfile.ZipFile(source) as archive:
        members = archive.namelist()
        for split, alias in [('test', 'test'), ('valid', 'val'), ('train', 'train')]:
            matches = [n for n in members if n.endswith(f'{split}/_annotations.coco.json')]
            if len(matches) != 1:
                raise ValueError(f'Expected one {split}/_annotations.coco.json; found {matches}')
            annotation_path = PurePosixPath(matches[0])
            data = json.loads(archive.read(str(annotation_path)))
            category_map = {c['id']: c['name'].strip().lower() for c in data['categories']}
            images = {im['id']: im for im in data['images']}
            if len(images) != len(data['images']):
                raise ValueError('Duplicate image IDs')
            annotations = defaultdict(list)
            counts = Counter()
            for ann in data['annotations']:
                if ann['image_id'] not in images:
                    raise ValueError('Annotation references an unknown image')
                name = category_map.get(ann['category_id'])
                if name not in ('fire', 'smoke'):
                    raise ValueError(f'Unexpected annotated class: {name}')
                if ann.get('iscrowd', 0):
                    raise ValueError('Crowd annotations require an explicit conversion policy')
                annotations[ann['image_id']].append(ann)
                counts[name] += 1
            report[alias] = {'images_original': len(images), 'objects_original': dict(counts)}
            stems = set()
            for im in images.values():
                filename = PurePosixPath(im['file_name'])
                if filename.is_absolute() or '..' in filename.parts:
                    raise ValueError('Unsafe image filename')
                # Flatten COCO paths deliberately; refuse stem collisions.
                if filename.stem in stems:
                    raise ValueError(f'Duplicate image stem in {alias}: {filename.stem}')
                stems.add(filename.stem)
                width, height = im['width'], im['height']
                if width <= 0 or height <= 0:
                    raise ValueError('Invalid image dimensions')
                member = str(annotation_path.parent / filename)
                image_bytes = archive.read(member)
                lines = []
                for ann in annotations[im['id']]:
                    class_id = ('fire', 'smoke').index(category_map[ann['category_id']])
                    if task == 'segment':
                        segments = ann.get('segmentation')
                        if not segments:
                            raise ValueError(
                                f'{split}/{filename}: segmentation is EMPTY. '
                                'Add reviewed polygon masks before segmentation training. '
                                'Use --task detect only if you want a separate box-only baseline.'
                            )
                        if not isinstance(segments, list) or len(segments) != 1:
                            raise ValueError('This converter supports single polygons only; '
                                             'RLE/multipart instances need mask-aware conversion.')
                        polygon = segments[0]
                        if len(polygon) < 6 or len(polygon) % 2:
                            raise ValueError('Polygon must contain at least three x,y pairs')
                        values = [v / (width if i % 2 == 0 else height)
                                  for i, v in enumerate(polygon)]
                        if any(not math.isfinite(v) or v < 0 or v > 1 for v in values):
                            raise ValueError('Polygon is outside image bounds')
                        area = abs(sum(polygon[i] * polygon[(i + 3) % len(polygon)]
                                       - polygon[(i + 2) % len(polygon)] * polygon[i + 1]
                                       for i in range(0, len(polygon), 2))) / 2
                        if area <= 0:
                            raise ValueError('Degenerate polygon')
                    else:
                        x, y, w, h = ann['bbox']
                        if not all(math.isfinite(v) for v in (x, y, w, h)):
                            raise ValueError('Non-finite bbox')
                        if w <= 0 or h <= 0 or min(x, y) < 0 or x+w > width or y+h > height:
                            raise ValueError('Invalid or out-of-image bbox')
                        values = [(x+w/2)/width, (y+h/2)/height, w/width, h/height]
                    lines.append(str(class_id) + ' ' + ' '.join(f'{v:.8f}' for v in values))
                rows.append(dict(split=alias, filename=filename.name, member=member,
                                 sha=hashlib.sha256(image_bytes).hexdigest(), labels=lines))

        # Preserve held-out evaluation sets; remove train images identical to either.
        heldout = {}
        for row in rows:
            if row['split'] != 'train':
                previous = heldout.get(row['sha'])
                if previous and previous['split'] != row['split']:
                    raise ValueError('Identical image in validation AND test; resolve split first')
                heldout[row['sha']] = row
        dropped, kept = [], []
        for row in rows:
            if row['split'] == 'train' and row['sha'] in heldout:
                dropped.append({'train': row['member'], 'heldout': heldout[row['sha']]['member']})
            else:
                kept.append(row)
        for row in kept:
            image_dir = output / 'images' / row['split']
            label_dir = output / 'labels' / row['split']
            image_dir.mkdir(parents=True, exist_ok=True)
            label_dir.mkdir(parents=True, exist_ok=True)
            (image_dir / row['filename']).write_bytes(archive.read(row['member']))
            (label_dir / Path(row['filename']).with_suffix('.txt')).write_text(
                '\n'.join(row['labels']) + ('\n' if row['labels'] else ''), encoding='utf-8')
    for split in report:
        report[split]['images_used'] = sum(r['split'] == split for r in kept)
    manifest = {'task': task, 'names': ['fire', 'smoke'], 'splits': report,
                'removed_train_duplicates': dropped,
                'note': 'Exact-byte duplicates only. Review scene/video leakage separately.'}
    (output / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    # JSON is valid YAML and avoids adding a dependency to this conversion script.
    config = {'path': output.as_posix(), 'train': 'images/train', 'val': 'images/val',
              'test': 'images/test', 'names': ['fire', 'smoke']}
    (output / 'data.yaml').write_text(json.dumps(config, indent=2), encoding='utf-8')
    print(json.dumps(manifest, indent=2))
    print('Data:', output / 'data.yaml')


## 5. แปลง COCO เป็นข้อมูลสำหรับ YOLO

ครั้งแรกจะสร้าง `DATA_DIR` หากมีข้อมูลเดิมแล้วจะตรวจว่า task ตรงกันและใช้ต่อ
หากเปลี่ยน ZIP หรือแก้ annotation ให้ตั้งชื่อ `DATA_DIR` ใหม่ก่อนรัน ไม่ใช้ label เก่าปนกับชุดใหม่
โหมด segment รับ single polygon ต่อ instance; RLE/multipart จะหยุดให้จัดการ mask ก่อน

In [ ]:
if DATA_DIR.exists():
    manifest_path = DATA_DIR / 'manifest.json'
    if not manifest_path.is_file():
        raise RuntimeError('โฟลเดอร์มีอยู่แต่ไม่มี manifest: เลือก DATA_DIR ใหม่')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest['task'] != TASK:
        raise RuntimeError('Task ไม่ตรงกับข้อมูลเดิม: เลือก DATA_DIR ใหม่')
    print('ใช้ข้อมูลที่แปลงไว้แล้ว — หากเปลี่ยน ZIP ให้เลือก DATA_DIR ใหม่')
else:
    prepare(ZIP_PATH, DATA_DIR, TASK)

DATA_YAML = DATA_DIR / 'data.yaml'
manifest = json.loads((DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))
display(Markdown('| ชุด | ภาพที่ใช้ |\n|---|---:|\n' + '\n'.join(
    f'| {split} | {info["images_used"]} |' for split, info in manifest['splits'].items())))
print('ตัดภาพ Train ซ้ำ:', len(manifest['removed_train_duplicates']), 'ภาพ')
print('พร้อมใช้:', DATA_YAML)


## 6. ดูภาพและ label ก่อน Train

กรอบ/ขอบเขตต้องตรงกับไฟหรือควัน ตรวจหลายภาพโดยเปลี่ยน `PREVIEW_INDEX`
สีแดง = fire, สีฟ้า = smoke

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
from PIL import Image

PREVIEW_INDEX = 0  # เปลี่ยนตัวเลขเพื่อดูภาพอื่น
train_images = sorted((DATA_DIR / 'images/train').iterdir())
image_path = train_images[PREVIEW_INDEX % len(train_images)]
image = Image.open(image_path).convert('RGB')
w, h = image.size
fig, ax = plt.subplots(figsize=(11, 7))
ax.imshow(image)
label_path = DATA_DIR / 'labels/train' / (image_path.stem + '.txt')
for line in label_path.read_text().splitlines():
    parts = line.split()
    class_id, values = int(parts[0]), list(map(float, parts[1:]))
    color = ['red', 'deepskyblue'][class_id]
    if TASK == 'detect':
        cx, cy, bw, bh = values
        x, y = (cx - bw/2)*w, (cy - bh/2)*h
        ax.add_patch(Rectangle((x, y), bw*w, bh*h, fill=False, edgecolor=color, linewidth=2))
    else:
        points = [(values[i]*w, values[i+1]*h) for i in range(0, len(values), 2)]
        ax.add_patch(Polygon(points, closed=True, facecolor=color, edgecolor=color, alpha=0.35))
        x, y = points[0]
    ax.text(x, y, ['fire', 'smoke'][class_id], color='white', backgroundcolor=color)
ax.set_title(image_path.name)
ax.axis('off')
plt.show()


## 7. Train โมเดล

เซลล์นี้เริ่มฝึกจริง ครั้งแรกจะดาวน์โหลด pretrained weights จึงต้องต่ออินเทอร์เน็ต
ใช้ `workers=0` สำหรับ Notebook บน Windows และไม่ cache ภาพทั้งหมดใน RAM
ไม่ใช้ test set เลือกโมเดล; ใช้ validation ระหว่างฝึก

เมื่อจบจะบอกที่อยู่ `best.pt` และ `last.pt` ชื่อ run ที่ซ้ำอาจถูกเติมเลขท้ายโดยอัตโนมัติ
ค่า confidence ที่โมเดลทำนายยังไม่ใช่ผลตัดสิน sensor fusion หรือสัญญาณ alarm ของทั้งระบบ

In [ ]:
if manifest['task'] != TASK:
    raise RuntimeError('Task changed: rerun configuration and preparation cells first')
MODEL_NAME = 'yolo11n-seg.pt' if TASK == 'segment' else 'yolo11n.pt'
model = YOLO(MODEL_NAME)
train_results = model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMAGE_SIZE,
    batch=BATCH, device=DEVICE, workers=0, cache=False,
    patience=25, seed=42, deterministic=True, amp=True,
    hsv_h=0.0, hsv_s=0.2, hsv_v=0.2, flipud=0.0,
    close_mosaic=min(10, EPOCHS), project=str(WORK_DIR / 'runs'), name=RUN_NAME,
)
RUN_DIR = Path(model.trainer.save_dir)
BEST_WEIGHTS = Path(model.trainer.best)
print('โฟลเดอร์ผล:', RUN_DIR)
print('โมเดลที่ดีที่สุด:', BEST_WEIGHTS)


## 8. ดูกราฟการฝึกและผล validation ใน Notebook

การฝึก 1 epoch ใช้ตรวจว่า pipeline ทำงานได้ ยังใช้สรุปความแม่นยำไม่ได้

In [ ]:
from IPython.display import Image as DisplayImage

for filename in ['results.png', 'val_batch0_pred.jpg', 'confusion_matrix.png']:
    path = RUN_DIR / filename
    if path.is_file():
        print(filename)
        display(DisplayImage(filename=str(path), width=950))


## 9. ประเมิน best.pt

เริ่ม `EVAL_SPLIT="val"` เพื่อวิเคราะห์และปรับโมเดล
เปลี่ยนเป็น `"test"` เมื่อจบการปรับแล้ว เพื่อประเมินข้อมูลที่กันไว้
งาน segment ต้องดูทั้ง box mAP และ mask mAP รวมผลรายคลาส fire/smoke
งาน detect ไม่มี mask mAP

In [ ]:
EVAL_SPLIT = 'val'
trained_model = YOLO(str(BEST_WEIGHTS))
metrics = trained_model.val(data=str(DATA_YAML), split=EVAL_SPLIT,
                            imgsz=IMAGE_SIZE, device=DEVICE, batch=BATCH, workers=0, plots=True)
print(f'Box mAP50: {metrics.box.map50:.4f}')
print(f'Box mAP50-95: {metrics.box.map:.4f}')
if TASK == 'segment':
    print(f'Mask mAP50: {metrics.seg.map50:.4f}')
    print(f'Mask mAP50-95: {metrics.seg.map:.4f}')


## 10. ทำนายภาพใหม่และดูกรอบ / mask

ตัวอย่างเริ่มจากภาพ validation หนึ่งภาพ เปลี่ยน `PREDICT_SOURCE` เป็นภาพของคุณหรือโฟลเดอร์ภาพได้
เซลล์นี้รองรับภาพนิ่งหรือโฟลเดอร์ภาพ ไม่ใช่กล้อง/วิดีโอสด
ใช้ `stream=True` เพื่อไม่สะสมผลทุกภาพในหน่วยความจำ และแสดงตัวอย่างไม่เกิน 4 ภาพ

In [ ]:
PREDICT_SOURCE = str(sorted((DATA_DIR / 'images/val').iterdir())[0])
# ตัวอย่าง: PREDICT_SOURCE = r'D:\my_test_images'
CONFIDENCE = 0.25  # ค่าเริ่มต้น; ปรับจาก validation

import cv2
import numpy as np
from datetime import datetime

PREDICT_DIR = WORK_DIR / 'predictions' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
PREDICT_DIR.mkdir(parents=True, exist_ok=False)
prediction_records = []
predictions = trained_model.predict(source=PREDICT_SOURCE, device=DEVICE,
                                    conf=CONFIDENCE, retina_masks=True, stream=True, save=False)
for image_index, result in enumerate(predictions):
    key = f'{image_index:05d}_{Path(result.path).stem}'
    overlay = result.plot()
    if not cv2.imwrite(str(PREDICT_DIR / f'{key}_overlay.jpg'), overlay):
        raise OSError('บันทึก overlay ไม่สำเร็จ')
    if image_index < 4:
        plt.figure(figsize=(11, 7))
        plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title(Path(result.path).name)
        plt.show()
    for i, box in enumerate(result.boxes):
        class_id = int(box.cls.item())
        record = dict(image=str(result.path), class_id=class_id,
                      class_name=result.names[class_id], confidence=float(box.conf.item()),
                      bbox_xyxy=box.xyxy[0].cpu().tolist())
        if result.masks is not None:
            mask = result.masks.data[i].cpu().numpy() > 0.5
            if mask.shape != result.orig_shape:
                raise ValueError('Mask ต้องมีขนาดเท่าภาพต้นฉบับ')
            mask_name = f'{key}_{i}_{record["class_name"]}.png'
            if not cv2.imwrite(str(PREDICT_DIR / mask_name), mask.astype(np.uint8)*255):
                raise OSError('บันทึก mask ไม่สำเร็จ')
            ys, xs = np.where(mask)
            record.update(mask_file=mask_name, area_pixels=int(mask.sum()),
                          centroid_xy=[float(xs.mean()), float(ys.mean())] if len(xs) else None)
        prediction_records.append(record)
(PREDICT_DIR / 'predictions.json').write_text(json.dumps(prediction_records, indent=2), encoding='utf-8')
print('บันทึกผล:', PREDICT_DIR)
print('วัตถุที่ตรวจพบ:', len(prediction_records))


## 11. บันทึกเวอร์ชัน environment เพื่อทำซ้ำ

ทำเมื่อ environment รันได้แล้ว บันทึกแพ็กเกจจริงที่ kernel นี้ใช้

In [ ]:
import subprocess

versions = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(WORK_DIR / 'installed_versions_notebook.txt').write_text(versions, encoding='utf-8')
print('บันทึกแล้ว:', WORK_DIR / 'installed_versions_notebook.txt')


## ทางเลือก: ฝึกต่อหลังหยุดกลางทาง

**ไม่ต้องรันเซลล์ด้านล่างในการฝึกปกติ** ใช้เฉพาะ run ที่หยุดก่อนครบ epochs
แก้ path ให้ตรงกับ `last.pt` ของ run เดิม และเปลี่ยน `RESUME_TRAINING=True`
resume ใช้ค่าจาก checkpoint เดิม จึงไม่ใช่การเปลี่ยน EPOCHS เพื่อเพิ่มรอบหลังฝึกจบแล้ว

In [ ]:
RESUME_TRAINING = False
LAST_WEIGHTS = WORK_DIR / 'runs' / RUN_NAME / 'weights' / 'last.pt'

if RESUME_TRAINING:
    assert LAST_WEIGHTS.is_file(), f'ไม่พบ checkpoint: {LAST_WEIGHTS}'
    model = YOLO(str(LAST_WEIGHTS))
    model.train(resume=True)
    RUN_DIR = Path(model.trainer.save_dir)
    BEST_WEIGHTS = Path(model.trainer.best)
    print('ฝึกต่อเสร็จแล้ว:', BEST_WEIGHTS)
else:
    print('ข้าม: ไม่ได้เปิด resume')


## ใช้ผลกับโปรเจกต์ต่ออย่างไร

- `best.pt`: โมเดลสำหรับทดลอง inference บนโน้ตบุ๊ก
- mask PNG: ค่า 255 คือพื้นที่ instance ที่ทำนายได้; ค่า 0 คือพื้นที่นอก instance นั้น
- `predictions.json`: กรอบ/class/confidence และ area/centroid เมื่อมี segmentation
- ค่า area เป็นพิกเซล ไม่ใช่ตารางเมตร และยังไม่ใช่ผล tracking ข้ามเฟรม
- ใช้ mask กับ `extract_depth_feature(depth_region, mask=...)` ใน `Camera/Phase2.py`
  เฉพาะเมื่อ RGB, mask และ aligned depth เป็นพิกัด/เฟรมเดียวกัน
- หากต้องแยกควันกับฝุ่นอย่างน่าเชื่อถือ ต้องประเมิน hard negatives และทำ sensor fusion ตามแผนโปรเจกต์ต่อ
- deployment บน Pi/Hailo ต้องทดสอบการ export, compilation และ postprocessing แยกจากการ Train

เมื่อเพิ่ม mask: ใช้กรอบเดิมช่วยร่างด้วย SAM ได้ แต่ตรวจแก้รูปร่างจริงก่อนฝึก
กำหนดนโยบายขอบควันจาง/ส่วนที่ทับซ้อนให้เหมือนกันทั้งชุด และแบ่งชุดตาม session/เหตุการณ์
ถ้าเป็นภาพที่บันทึกจาก `Camera/phase3.py` อย่าสุ่มเฟรมที่คล้ายกันจาก session เดียวไปคนละชุด

แหล่งอ้างอิง:
[SeniorINC](https://github.com/Peanut2547/SeniorINC) ·
[แนวทางโปรเจกต์](https://github.com/Peanut2547/SeniorINC/blob/main/Document_theory/SPTAnalysis_Ver01.md) ·
[YOLO11](https://docs.ultralytics.com/models/yolo11/) ·
[Segmentation labels](https://docs.ultralytics.com/datasets/segment/) ·
[PyTorch](https://pytorch.org/get-started/locally/)

อ้างอิงการอ่าน repo เดิม ณ commit `7f7c9219cb2c4ed7ce151e4f5c75024fe4ecc32e`